Neuronal net with cylinical split data

In [3]:
# ============================================================
# 0) Imports
# ============================================================
%pip -q install tensorflow scikit-learn joblib

import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
import joblib

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, InputLayer
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam


Note: you may need to restart the kernel to use updated packages.


2025-12-17 14:39:53.267899: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-17 14:39:53.268307: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-17 14:39:53.320447: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-17 14:39:54.577198: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur

In [4]:
# ============================================================
# 1) Load your pre-split data
# ============================================================
TRAIN_PATH = "/workspaces/bakery_prediction/0_DataPreparation/Split_data/03_cylindical/train_data.csv"
VAL_PATH   = "/workspaces/bakery_prediction/0_DataPreparation/Split_data/03_cylindical/val_data.csv"
TEST_PATH  = "/workspaces/bakery_prediction/0_DataPreparation/Split_data/03_cylindical/test_data.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

# if you have a date column, parse it (optional for the NN, useful for checks)
if "Datum" in train_df.columns:
    train_df["Datum"] = pd.to_datetime(train_df["Datum"])
    val_df["Datum"]   = pd.to_datetime(val_df["Datum"])
    test_df["Datum"]  = pd.to_datetime(test_df["Datum"])

print(train_df.shape, val_df.shape, test_df.shape)
train_df.head()


(7493, 23) (1841, 23) (1830, 23)


,id,Datum,Warengruppe,Umsatz,KielerWoche,Bewoelkung,Temperatur,Windgeschwindigkeit,Woche,Monat,...,Ferien,sunny,cloudy,rainy,thunderstorm,is_weekend,sin_Monat,cos_Monat,sin_Wochentag,cos_Wochentag
0,1307011,2013-07-01,1,148.828353,0,6,17.8375,15,27,7,...,0,0,1,0,0,0,-0.5,-0.866025,0.781831,0.62349
1,1307012,2013-07-01,2,535.856285,0,6,17.8375,15,27,7,...,0,0,1,0,0,0,-0.5,-0.866025,0.781831,0.62349
2,1307013,2013-07-01,3,201.198426,0,6,17.8375,15,27,7,...,0,0,1,0,0,0,-0.5,-0.866025,0.781831,0.62349
3,1307014,2013-07-01,4,65.890169,0,6,17.8375,15,27,7,...,0,0,1,0,0,0,-0.5,-0.866025,0.781831,0.62349
4,1307015,2013-07-01,5,317.475875,0,6,17.8375,15,27,7,...,0,0,1,0,0,0,-0.5,-0.866025,0.781831,0.62349


In [5]:
# ============================================================
# 2) Define target + feature columns
#    (adjust drop list to your columns)
# ============================================================
TARGET_COL = "Umsatz"

drop_cols = []
for c in ["id", "Datum"]:
    if c in train_df.columns:
        drop_cols.append(c)

# IMPORTANT: avoid leakage for forecasting:
# if Umsatz_roll is computed using future values, drop it.
# If it's computed only from past values at that date, it can stay.
if "Umsatz_roll" in train_df.columns:
    drop_cols.append("Umsatz_roll")

feature_cols = [c for c in train_df.columns if c not in drop_cols + [TARGET_COL]]

X_train = train_df[feature_cols].copy()
y_train = train_df[TARGET_COL].copy()

X_val   = val_df[feature_cols].copy()
y_val   = val_df[TARGET_COL].copy()

X_test  = test_df[feature_cols].copy()
y_test  = test_df[TARGET_COL].copy()

print("n_features:", len(feature_cols))


n_features: 20


In [6]:
# ============================================================
# 3) Scale features (fit ONLY on train)
# ============================================================
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

os.makedirs("3_Model/class file/pickle_data", exist_ok=True)
joblib.dump(scaler, "3_Model/class file/pickle_data/scaler.joblib")
pd.to_pickle(feature_cols, "3_Model/class file/pickle_data/feature_cols.pkl")


In [7]:
# ============================================================
# 4) Build + train neural net
# ============================================================
tf.random.set_seed(42)

model = Sequential([
    InputLayer(shape=(X_train_s.shape[1],)),
    Dense(256, activation="relu"),
    BatchNormalization(),
    Dropout(0.2),

    Dense(128, activation="relu"),
    BatchNormalization(),
    Dropout(0.2),

    Dense(64, activation="relu"),
    Dense(1)
])

model.compile(
    optimizer=Adam(1e-3),
    loss="mse",
    metrics=["mae"]
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", patience=5, factor=0.5, min_lr=1e-6),
]

history = model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=100,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)

model.save("3_Model/class file/python_model.h5")


Epoch 1/100


2025-12-17 14:49:04.341546: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 63600.9844 - mae: 205.1142 - val_loss: 53620.8359 - val_mae: 192.3481 - learning_rate: 0.0010
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 58081.6133 - mae: 193.5310 - val_loss: 47861.9570 - val_mae: 179.4258 - learning_rate: 0.0010
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 47983.0586 - mae: 170.9017 - val_loss: 37696.5430 - val_mae: 153.0980 - learning_rate: 0.0010
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 36707.4297 - mae: 146.3748 - val_loss: 28009.7148 - val_mae: 128.4946 - learning_rate: 0.0010
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 29788.3789 - mae: 132.6147 - val_loss: 22686.1680 - val_mae: 117.9590 - learning_rate: 0.0010
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 26030.1094 - mae: 122.4564 - val_loss: 19994.4746 - val_mae: 111.1608 - learning_rate: 0.0010
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 23134.3203 - mae: 115.0011 - val_l

In [8]:
# ============================================================
# 5) Evaluate overall + per Warengruppe (if Group_1..Group_6 exist)
# ============================================================
def mape(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

test_pred = model.predict(X_test_s).reshape(-1)
print("Test MAE:", np.mean(np.abs(y_test.values - test_pred)))
print("Test MAPE:", mape(y_test.values, test_pred))

group_cols = [c for c in feature_cols if c.startswith("Group_")]
if group_cols:
    tmp = test_df[["Umsatz"] + group_cols].copy()
    tmp["pred"] = test_pred
    for g in group_cols:
        part = tmp[tmp[g] == 1]
        if len(part) > 0:
            print(g, "MAPE:", mape(part["Umsatz"], part["pred"]))


58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Test MAE: nan
Test MAPE: nan
